-  시작전 설치

In [ ]:
!pip install -q Korpora
!pip install -q konlpy
!pip install -q transformers

### 1.데이터셋 불러오기
- Korpora를 통해 불러오는 방법 (kcbert) , nsmc도가능하긴함
- 링크를 통해 불러오는 방법 (nsmc)

#### 1.1 kcbert

In [ ]:
from Korpora import Korpora
Korpora.corpus_list()

In [ ]:
corpus = Korpora.load("kcbert")

In [ ]:
len(corpus.train) # 설치 시 입력한 갯수만큼 불러온다.
# corpus 말뭉치, 전체 학습 데이터 셋

In [ ]:
#문장들 길이 파악
import numpy as np

sentence_len = [len(corpus.train[i]) for i in range(len(corpus.train[:]))]
print(f"제일 긴 문장길이 : {max(sentence_len)}")
print(f"제일 짧은 문장길이 : {min(sentence_len)}")
print(f'평균 문장길이 : {np.sum(sentence_len)/len(sentence_len)}')

In [ ]:
import matplotlib.pyplot as plt

plt.hist(sentence_len)
plt.xlabel("sentence length")
plt.ylabel("Frequency")

plt.show()

#### 1-2.nsmc

In [ ]:
import urllib.request
import pandas as pd

# 데이터 로드
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="test.txt")

train_df = pd.read_table('train.txt')
test_df = pd.read_table('test.txt')

In [ ]:
train_df.tail()

In [ ]:
corpus2 = Korpora.load("nsmc") #korpora로도 가능하다.

In [ ]:
len(corpus2.train) #15만개 다 불러옴

In [ ]:
len(corpus2.test) #5만개 다 불러옴

In [ ]:
corpus2.train[1000]

In [ ]:
# train data 문장길이 체크
import numpy as np

document_len = [len(train_df.document[i]) for i in range(10000)]
print(f"train 중 제일 긴 문장길이 : {max(document_len)}")
print(f"train 중 제일 짧은 문장길이 : {min(document_len)}")
print(f'train 평균 문장길이 : {np.sum(document_len)/len(document_len)}')

In [ ]:
import matplotlib.pyplot as plt

plt.hist(document_len)
plt.xlabel("sentence length")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# test data 문장길이 체크
import numpy as np

document_len_test = [len(test_df.document[i]) for i in range(5000)]
print(f"test 중 제일 긴 문장길이 : {max(document_len_test)}")
print(f"test 중 제일 짧은 문장길이 : {min(document_len_test)}")
print(f'test 평균 문장길이 : {np.sum(document_len_test)/len(document_len_test)}')

In [ ]:
import matplotlib.pyplot as plt

plt.hist(document_len_test)
plt.xlabel("sentence length")
plt.ylabel("Frequency")

plt.show()

### 2.Word2Vec 연습해보기

- gensim에서 사전학습된 모델 불러와서 해보기
- 직접 학습시킨 후 해보기

#### 2-1.CBOW 로 보는 W2V

In [ ]:
import torch
import torch.nn as nn

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        # vocab_size: 단어 사전의 크기(총 단어 개수).
        # embed_dim: 각 단어를 몇 차원 벡터로 표현할지(예: 100차원)
        super(CBOW, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)   # Embedding layer
        # 단어 인덱스를 입력하면 해당 단어를 벡터(embedding)로 변환해줌.
        # 예: "apple"(단어 번호 2) → [0.3, -0.1, 0.5, ...] (차원=embed_dim).
        self.linear = nn.Linear(embed_dim, vocab_size)         # Projection layer
        # 문맥(여러 단어 벡터 평균)을 받아서 다음 단어 예측 확률로 변환.
        # 출력 차원은 vocab_size (모든 단어에 대한 확률을 구하기 때문).

    def forward(self, context_idxs):
        # context_idxs: 여러 단어 인덱스 (ex. [2, 5, 7, 8])
        embeds = self.embedding(context_idxs)   # (N, D)
        # 임베딩 변환: context_idxs에 해당하는 단어들을 embedding으로 바꿈
        # N = 문맥 단어 개수 (예: 4개), D = 임베딩 차원 수 (예: 100)
        context_vec = embeds.mean(dim=0)        # 평균 (D,)
        # 문맥벡터 평균 >> 결과 크기: (D,) → 1개의 문맥 표현.
        out = self.linear(context_vec)          # (V,)
        # 문맥 벡터를 Linear에 통과시켜 어떤 단어가 중심 단어일지 점수(logit) 계산.
        # 출력 크기: (V,) → 단어 사전 크기만큼의 점수.
        log_probs = torch.log_softmax(out, dim=-1)
        return log_probs
        # 모든 단어 점수를 확률로 변환 (softmax).
        # 로그를 취해 log-probability로 변환 (log_softmax).
        # 확률 안정성과 학습 효율성을 위해 log_softmax 사용.

In [ ]:
# vocab: {"I":0, "like":1, "playing":2, "football":3, "very":4, "much":5}
vocab_size = 6
embed_dim = 10
model = CBOW(vocab_size, embed_dim)

# 문장: "I like playing football very much"
# 중심 단어: "playing"(2)
# context: ["I"(0), "like"(1), "football"(3), "very"(4)]
context_idxs = torch.tensor([0,1,3,4], dtype=torch.long)
target = torch.tensor([2])  # 예측할 중심 단어 "playing"

print(context_idxs)
print(target)

- forward 과정 뜯어보기

In [ ]:
context_idxs
# [[1,0,0,0,0,0],[0,1,0,0,0,0],[0,0,0,1,0,0],[0,0,0,0,1,0]]

In [ ]:
embeds = model.embedding(context_idxs)   # (4, 10)
# -> 각 단어(0,1,3,4)를 10차원 벡터로 변환

print(embeds)

In [ ]:
context_vec = embeds.mean(dim=0)         # (10,)
# -> context 벡터 평균

print(context_vec)

In [ ]:
out = model.linear(context_vec)          # (6,)
# -> vocab 크기만큼 projection
print(out)

In [ ]:
log_probs = torch.log_softmax(out, dim=-1) # (6,)
# -> 중심 단어가 될 확률 분포

print(log_probs)

In [ ]:
torch.argmax(log_probs)

#### 2-2.사전학습된 w2v 사용

In [ ]:
!pip install gensim

In [ ]:
#사전학습된 word2vec model list
import gensim.downloader

print(list(gensim.downloader.info()['models'].keys()))

In [ ]:
# gensim.downloader.info()['models']

In [ ]:
# model : glove-twitter-25
glove_vectors = gensim.downloader.load('glove-twitter-25')

In [ ]:
glove_vectors.index_to_key

In [ ]:
len(glove_vectors.index_to_key)

In [ ]:
glove_vectors.index_to_key[76908]

In [ ]:
glove_vectors.get_index('kimchi') # 120만개  [76908] -> model -> dim 25 vector / linear(1=input_size,25) =>

In [ ]:
glove_vectors.get_vector('kimchi')

In [ ]:
glove_vectors.most_similar('lunch',topn=5)

In [ ]:
glove_vectors.most_similar('dog',topn=5)

In [ ]:
glove_vectors.similarity('kimchi','bulgogi')

#### 2-3.생짜로 된 word2vec 사용

In [ ]:
from gensim.models import Word2Vec

In [ ]:
sentences = [['cat','say','meow'],['dog','say','woof']]
model = Word2Vec(sentences,vector_size=50,window=2, min_count=1)
# vector_size : 임베딩 벡터의 크기(차원)
# window : 주변 단어를 고려하는 범위
# min_count : 임베딩 시킬 단어에 포함시키는 임계값
# >> 즉 최소 1번은 나와야 함

In [ ]:
print("임베딩 벡터: ", model.wv['dog'])
print("임베딩 사이즈: ",len(model.wv['dog']))

In [ ]:
model.wv.similarity('dog','woof')

In [ ]:
model.wv['dog']

### 3. Tokenizer 사용해보기
- 형태소분석기 사용

#### 3-1. 형태소 분석기(Okt-Open Korea Text, Kkma)

In [ ]:
from konlpy.tag import Okt , Kkma
Okt_tokenizer = Okt()
Kkma_tokenizer = Kkma()

text = "아버지가 방에 들어가신다. 이순신은 조선시대 무관이다."

In [ ]:
Okt_tokenizer.morphs(text)

In [ ]:
Okt_tokenizer.pos(text)

In [ ]:
Okt_tokenizer.nouns(text)

In [ ]:
Kkma_tokenizer.morphs(text)

In [ ]:
Kkma_tokenizer.pos(text)

In [ ]:
Kkma_tokenizer.nouns(text)

In [ ]:
!pip install soynlp

In [ ]:
from soynlp.tokenizer import LTokenizer
from soynlp.noun import LRNounExtractor_v2

noun_extractor = LRNounExtractor_v2()
nouns = noun_extractor.train_extract(["안녕하세요. 자연어 처리는 재미있습니다."])
print(nouns)

tokenizer = LTokenizer()
tokens = tokenizer.tokenize("안녕하세요. 자연어 처리는 재미있습니다.")
print(tokens)

In [ ]:
# eos